# 📖 Notebook 2: Pods & Deployments — Running Your Applications

Now that you have a cluster, it's time to run real workloads.
In this notebook, you'll create a pod, package sample services into your cluster, and learn how Deployments keep your applications healthy.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Explain what a pod is and why it is the smallest deployable Kubernetes unit
- Create a pod from YAML
- Build sample application images directly into minikube
- Explain what a Deployment does for you
- Scale a Deployment to multiple replicas
- Perform a rolling update and watch the rollout
- Roll back to the previous version if needed
- Add resource requests and limits, and predict the pod's QoS class
- Say what actually happens when a container exceeds its CPU limit vs its memory limit
- Configure liveness, readiness and startup probes, and explain what each one does
- Recover from a rollout that never becomes healthy

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

# `docker` is needed here and not in notebook 1: this notebook BUILDS the three
# sample images and pushes them into the cluster's own image store.
REQUIRED = ['docker', 'minikube', 'kubectl']
INSTALL_HINTS = {
    'docker': 'https://docs.docker.com/get-docker/  (or `brew install --cask docker`)',
    'minikube': 'https://minikube.sigs.k8s.io/docs/start/  (or `brew install minikube`)',
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Assertion helpers ────────────────────────────────────────────────────
# A `!kubectl ...` line that exits non-zero does NOT fail a notebook cell -- it
# just prints red text that scrolls away. Every claim this notebook makes below
# is therefore checked in Python as well, so the notebook stops at the first
# thing that stopped being true instead of printing ten more cells of noise.
import json
import subprocess
import time

NS = "k8s-lab"


def kget(*args, ns=NS):
    """kubectl get <args> -n <ns> -o json, parsed."""
    cmd = ["kubectl", "get", *args, "-o", "json"]
    if ns:
        cmd += ["-n", ns]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def pods(selector, ns=NS):
    return kget("pods", "-l", selector, ns=ns)["items"]


def is_ready(pod):
    conds = {c["type"]: c["status"] for c in pod["status"].get("conditions", [])}
    return conds.get("Ready") == "True"


def terminating(pod):
    """A pod being deleted keeps reporting Ready until its grace period expires.
    Counting those is the classic source of flaky "expected 3, got 4" checks."""
    return "deletionTimestamp" in pod["metadata"]


def ready_pods(selector, ns=NS):
    """Pods whose Ready condition is True -- the same set a Service would route to."""
    return [p for p in pods(selector, ns=ns) if is_ready(p) and not terminating(p)]


def endpoint_ips(name, ns=NS):
    """Addresses currently behind a Service. Empty == nothing is receiving traffic."""
    r = subprocess.run(["kubectl", "get", "endpoints", name, "-n", ns, "-o", "json"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return []
    return [a["ip"]
            for subset in json.loads(r.stdout).get("subsets", []) or []
            for a in subset.get("addresses", []) or []]


def wait_until(predicate, timeout=180, interval=3, what="condition"):
    """Poll until predicate() is truthy. Returns its value; raises on timeout."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    raise AssertionError(f"timed out after {timeout}s waiting for {what}")


def settled_pods(selector, count, timeout=180, ns=NS):
    """Wait until exactly `count` Ready, non-terminating pods match, then return them.
    `kubectl rollout status` returns the moment the new ReplicaSet is available,
    which can be a second or two before the old pods finish terminating."""
    return wait_until(
        lambda: (lambda r: r if len(r) == count else None)(ready_pods(selector, ns=ns)),
        timeout=timeout, interval=2,
        what=f"exactly {count} Ready pods matching {selector}",
    )


print("assertion helpers ready")

## 🛠️ Setup

### Prerequisites

**Notebook 1 must be finished**, and the minikube cluster it created must still be
running. Everything below talks to that cluster.

Docker must also be running: this notebook builds the three sample images with
`docker build` and then loads them into the cluster's own image store.

### What this notebook leaves behind

Notebooks 3 to 10 all expect a namespace called **`k8s-lab`** containing three
deployments — `api-gateway`, `user-service`, `order-service`. This notebook creates
them. Its cleanup cell deliberately keeps them.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

### Exercise

Verify that the cluster is running and that `kubectl` can see your node.

In [ ]:
!minikube status
!kubectl get nodes

## 🗂️ Create the `k8s-lab` Namespace

A **namespace** is a name scope. Two objects can share a name as long as they live in
different namespaces, and most `kubectl` commands are namespace-scoped — which is why
you will see `-n k8s-lab` on nearly every command from here on. Forget the `-n` and
kubectl silently looks in `default` instead and reports `No resources found`. That is
the single most common "but I just created it!" moment in Kubernetes.

The namespace manifest also carries **Pod Security Standards** labels
(`pod-security.kubernetes.io/enforce: baseline`, `warn: restricted`). `baseline` rejects
genuinely dangerous pods; `warn: restricted` only prints a warning. Our sample images run
as root, so expect a yellow `Warning: would violate PodSecurity "restricted"` line on
every apply. That warning is the lab working as intended — notebook 6 explains it.

In [ ]:
# `apply` is idempotent: running it on an existing namespace is a no-op, not an error.
# That is why every cell in this series uses `apply` instead of `create`.
!kubectl apply -f ../manifests/namespace.yaml
!kubectl get namespace k8s-lab --show-labels

## 🧱 Build the Sample Application Images — Into the Cluster

The lab folder includes three small sample services:
- `user-service`
- `order-service`
- `api-gateway`

### The trap: a local `docker build` is invisible to your cluster

This is the first thing that bites everyone running Kubernetes locally, and it produces a
failure that looks like it must be a typo:

```text
$ docker build -t k8s-lab/user-service:latest apps/user-service
$ docker images | grep user-service          # it is RIGHT THERE
k8s-lab/user-service   latest   bbf22ca79642   4 seconds ago
$ kubectl get pods -n k8s-lab
user-service-6d4f...   0/1   ImagePullBackOff
```

Two separate things are going wrong at once:

1. **A local cluster has its own image store.** minikube's node is a container with its
   own container runtime, and `kind`'s is too. Your laptop's Docker daemon and the
   cluster's runtime are different registries that happen to live on the same machine.
   Docker Desktop's *built-in* Kubernetes is the exception — there the daemon is shared,
   which is why advice that works for one person fails for the next.
2. **`:latest` forces a registry pull.** Kubernetes defaults `imagePullPolicy` to `Always`
   for any image tagged `:latest` (and `IfNotPresent` for every other tag). So even once
   the image *is* in the cluster's store, the kubelet will ignore it, go ask Docker Hub
   for `docker.io/k8s-lab/user-service:latest`, and fail. Every manifest in this lab that
   uses these images sets `imagePullPolicy: IfNotPresent` for exactly this reason.

The next cell handles part 1 for whichever local cluster you are on: it builds with plain
`docker build` and then pushes the result into the cluster's own store — `minikube image
load` on minikube, `kind load docker-image` on kind, nothing at all on Docker Desktop.
Then it *verifies* the images are visible to the cluster rather than assuming it.

### Exercise

Build the sample images and load them into your cluster.

In [ ]:
import json
import shutil
import subprocess

IMAGES = {
    "k8s-lab/user-service:latest": "../apps/user-service",
    "k8s-lab/order-service:latest": "../apps/order-service",
    "k8s-lab/api-gateway:latest": "../apps/api-gateway",
}


def sh(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)


def cluster_flavour():
    """Which local Kubernetes are we on? Decides how images get in."""
    ctx = sh(["kubectl", "config", "current-context"]).stdout.strip()
    nodes = sh(["kubectl", "get", "nodes", "-o",
                "jsonpath={range .items[*]}{.metadata.name} {end}"]).stdout.split()
    if ctx == "minikube" or "minikube" in nodes:
        return "minikube"
    if ctx.startswith("kind-") or any(n.endswith("-control-plane") for n in nodes):
        return "kind"
    if ctx in ("docker-desktop", "docker-for-desktop"):
        return "docker-desktop"
    return "unknown"


FLAVOUR = cluster_flavour()
print("cluster flavour:", FLAVOUR)


def images_in_cluster():
    """Image names the CLUSTER's runtime can see -- not your laptop's."""
    if FLAVOUR == "minikube":
        return sh(["minikube", "image", "ls"]).stdout
    if FLAVOUR == "kind":
        node = sh(["kubectl", "get", "nodes", "-o",
                   "jsonpath={.items[0].metadata.name}"]).stdout.strip()
        return sh(["docker", "exec", node, "crictl", "images"]).stdout
    # Docker Desktop's Kubernetes shares the host daemon.
    return sh(["docker", "image", "ls", "--format", "{{.Repository}}:{{.Tag}}"]).stdout


def build_and_load(tag, context):
    """Build on the host, then put the image where the cluster's kubelet can see it."""
    print(f"\n--- building {tag} ---")
    b = sh(["docker", "build", "-t", tag, context])
    if b.returncode != 0:
        raise RuntimeError(f"docker build failed for {tag}:\n{b.stderr[-1500:]}")
    print(b.stderr.strip().splitlines()[-1] if b.stderr.strip() else "built")

    if FLAVOUR == "minikube":
        print("  -> minikube image load")
        loaded = sh(["minikube", "image", "load", tag])
    elif FLAVOUR == "kind":
        print("  -> kind load docker-image")
        loaded = sh(["kind", "load", "docker-image", tag]) if shutil.which("kind") else None
    else:
        # Docker Desktop's Kubernetes shares the host daemon; nothing to copy.
        print("  -> shared daemon, nothing to load")
        loaded = None
    if loaded is not None and loaded.returncode != 0:
        raise RuntimeError(f"loading {tag} into the cluster failed:\n{loaded.stderr[-1000:]}")


for tag, context in IMAGES.items():
    build_and_load(tag, context)

# Verify rather than assume. This is the check whose absence turns a five-second
# fix into an hour of `kubectl describe`.
listing = images_in_cluster()
missing = [t for t in IMAGES if t.split(":")[0] not in listing]
assert not missing, (
    f"built but NOT visible to the cluster: {missing}\n"
    f"cluster flavour detected as {FLAVOUR!r}; images the cluster can see:\n{listing}"
)
print(f"\n✅ all {len(IMAGES)} images are in the cluster's own image store")

## 📦 What is a Pod?

A **pod** is the smallest thing Kubernetes schedules.
It usually contains one application container, but it can hold multiple containers that need to stay together.

Analogy: if a **container** is one worker, a **pod** is that worker plus their desk, badge, and local tools.
Kubernetes does not schedule the worker alone; it schedules the whole desk setup as one unit.

```text
┌──────────────────────────────┐
│            Pod               │
│                              │
│  ┌────────────────────────┐  │
│  │ user-service container │  │
│  └────────────────────────┘  │
│  shared IP + shared volumes  │
└──────────────────────────────┘
```

We'll save our YAML in the notebook folder so you can inspect it later.

### Exercise

Write a pod manifest.

In [ ]:
%%writefile ./pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: hello-pod
  namespace: k8s-lab
  labels:
    app: hello-pod
spec:
  containers:
    - name: hello-pod
      image: nginx:1.27
      ports:
        - containerPort: 80

In [ ]:
!kubectl apply -f ./pod.yaml
!kubectl wait --for=condition=Ready pod/hello-pod -n k8s-lab --timeout=120s
!kubectl get pods -n k8s-lab
!kubectl describe pod hello-pod -n k8s-lab
!kubectl logs hello-pod -n k8s-lab

pod = kget("pod", "hello-pod")
assert pod["status"]["phase"] == "Running", \
    f"hello-pod is {pod['status']['phase']}; read the Events in the describe output above"
print("\n✅ hello-pod is Running")

## 🚚 What is a Deployment?

A **Deployment** manages pods for you.
You tell Kubernetes the desired state — for example, "keep 3 copies of `user-service` running" — and the Deployment keeps trying to make that true.

This gives you three huge benefits:
- **Self-healing**: if a pod dies, Kubernetes creates a replacement
- **Scaling**: you can raise or lower the replica count
- **Rolling updates**: Kubernetes replaces old pods gradually instead of all at once

```text
Deployment
   │
   ▼
ReplicaSet
   │
   ├── Pod 1
   ├── Pod 2
   └── Pod 3
```

### Exercise

Write a Deployment manifest for `user-service`.

In [ ]:
%%writefile ./user-service-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: user-service
  namespace: k8s-lab
spec:
  replicas: 1
  # `selector` is how the Deployment finds the pods it owns. It MUST match the
  # labels in `template.metadata.labels` below, and it is immutable after creation.
  # A selector that matches nothing gives you a Deployment that endlessly creates
  # pods it does not recognise -- one of the nastier silent failures in Kubernetes.
  selector:
    matchLabels:
      app: user-service
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
  template:
    metadata:
      labels:
        app: user-service
    spec:
      containers:
        - name: user-service
          image: k8s-lab/user-service:latest
          # Without this the kubelet would try to PULL k8s-lab/user-service
          # from Docker Hub, because `:latest` defaults the policy to Always.
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8001

In [ ]:
!kubectl apply -f ./user-service-deployment.yaml
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=180s
!kubectl get deployments -n k8s-lab
!kubectl get pods -l app=user-service -n k8s-lab -o wide

dep = kget("deployment", "user-service")
assert dep["status"].get("readyReplicas") == 1, (
    f"expected 1 ready replica, got {dep['status'].get('readyReplicas')}. "
    "If the pod is in ImagePullBackOff, the image never made it into the cluster -- "
    "re-run the build cell above."
)
print("\n✅ 1/1 replicas ready")

## 📈 Scaling Replicas

If one pod can handle some traffic, then multiple replicas can handle more traffic and give you better availability.
Scaling a Deployment changes the desired number of pods.

### Exercise

Scale `user-service` from 1 replica to 3 replicas.

In [ ]:
!kubectl scale deployment user-service -n k8s-lab --replicas=3
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=180s
!kubectl get deployments -n k8s-lab
!kubectl get pods -l app=user-service -n k8s-lab

dep = kget("deployment", "user-service")
assert dep["spec"]["replicas"] == 3, "desired replicas was not raised to 3"
assert dep["status"].get("readyReplicas") == 3, \
    f"only {dep['status'].get('readyReplicas')} of 3 replicas are ready"
print("\n✅ 3/3 replicas ready")

## 🔄 Rolling Updates

A rolling update replaces pods **gradually** instead of all at once. Two numbers control
exactly how gradually, and they are the two knobs you will actually tune in production:

| Field | Meaning | Our value |
|---|---|---|
| `maxSurge` | How many pods **above** `replicas` may exist during the rollout | `1` |
| `maxUnavailable` | How many pods **below** `replicas` you tolerate during the rollout | `0` |

Both accept a count (`1`) or a percentage (`25%`). The pair we set in the manifest —
`maxSurge: 1, maxUnavailable: 0` — is the safe default for a request-serving service:
Kubernetes brings a *new* pod up and waits for it to become **Ready** before it removes
an old one, so capacity never dips below `replicas`. The cost is one extra pod's worth of
CPU and memory for the duration of the rollout.

The opposite setting, `maxSurge: 0, maxUnavailable: 1`, uses no extra capacity but runs
you one pod short during the rollout. That is the right choice for a batch worker and the
wrong choice for a service that is already at its capacity limit.

> **The part everyone misses**: "waits for it to become Ready" means *waits for the
> readiness probe*. With no readiness probe, a pod is Ready the moment its container
> process starts — before the app has loaded config, opened its database pool, or bound
> its port. `maxUnavailable: 0` then guarantees you nothing at all, because Kubernetes is
> measuring the wrong thing. We fix that further down.

For a safe demo, we'll build the same app code with a new image tag called `v2`.
Even if the code is unchanged, the new tag still lets you see the rollout process.

### Exercise

Build a new image tag, update the Deployment, and watch Kubernetes finish the rollout.

In [ ]:
# Same source, new tag. `:v2` is not `:latest`, so its default imagePullPolicy is
# already IfNotPresent -- the local copy is used without argument.
build_and_load("k8s-lab/user-service:v2", "../apps/user-service")

!kubectl set image deployment/user-service -n k8s-lab user-service=k8s-lab/user-service:v2
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=180s
!kubectl get pods -l app=user-service -n k8s-lab
!kubectl rollout history deployment/user-service -n k8s-lab

running = settled_pods("app=user-service", 3)
tags = {c["image"] for pod in running for c in pod["spec"]["containers"]}
assert tags == {"k8s-lab/user-service:v2"}, \
    f"rollout finished but pods are still running {tags}"
print(f"\n✅ all 3 pods are on {tags.pop()}")

## ⏪ Failure First: a Rollout That Never Finishes

Rolling back is only interesting once you have something to roll back *from*. So let's
break it on purpose: point the Deployment at an image tag that does not exist.

Watch what Kubernetes does — and, more importantly, what it does **not** do. It will not
tear down the working pods. `maxUnavailable: 0` means the old ReplicaSet keeps serving
while the new pod sits in `ImagePullBackOff` forever. Your users see nothing. The rollout
just never completes.

That is why `kubectl rollout status` takes a `--timeout`: in CI it is the thing that turns
"the deploy silently hung" into "the deploy failed".

### Exercise

Deploy a broken image, watch the rollout stall, then undo it.

In [ ]:
# Deliberately break the rollout: this tag was never built.
!kubectl set image deployment/user-service -n k8s-lab user-service=k8s-lab/user-service:does-not-exist

# This will NOT succeed. It exits non-zero after 60s -- that is the point of the demo.
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=60s || echo ">>> rollout did not complete within 60s (expected)"

print()
# The old pods are still Running and still serving. Only the new one is stuck.
!kubectl get pods -l app=user-service -n k8s-lab

# --- the three claims this section makes, checked ---
# One snapshot, so "ready" and "stuck" are two halves of the same list rather
# than two kubectl calls that saw the cluster at different moments.
snapshot = [p for p in pods("app=user-service") if not terminating(p)]
serving = [p for p in snapshot if is_ready(p)]
stuck = [p for p in snapshot if not is_ready(p)]

# 1. The rollout did NOT complete.
dep = kget("deployment", "user-service")
assert dep["status"].get("updatedReplicas", 0) < 3, \
    "the broken rollout somehow completed -- does that tag exist after all?"

# 2. Capacity never dipped: maxUnavailable: 0 kept all 3 old pods serving.
assert len(serving) == 3, (
    f"only {len(serving)} pods are still Ready -- maxUnavailable: 0 was supposed to "
    "guarantee 3. Users would be seeing errors right now."
)
assert all(c["image"] == "k8s-lab/user-service:v2"
           for pod in serving for c in pod["spec"]["containers"]), \
    "the pods still serving should be the previous, working version"

# 3. Exactly one extra pod exists (maxSurge: 1) and it is wedged on the image.
assert len(stuck) == 1, f"expected exactly 1 surge pod, found {len(stuck)}"
waiting = [cs["state"].get("waiting", {}).get("reason")
           for cs in stuck[0]["status"].get("containerStatuses", [])]
assert any(r in ("ImagePullBackOff", "ErrImagePull") for r in waiting), \
    f"the stuck pod is not failing on the image; it says {waiting}"

print(f"\n✅ reproduced: 3 old pods still Ready, 1 surge pod in {waiting[0]}")

Read the output above carefully:

- Three pods are still `Running` and `1/1` — the old, working version. Traffic is fine.
- One extra pod is `ImagePullBackOff` — that is `maxSurge: 1` at work.
- The Deployment's `updatedReplicas` is stuck at **1 of 3**: one pod on the new template
  was created, it never became Ready, and because `maxUnavailable: 0` forbids removing a
  working pod to make room, the rollout can never take another step. It does not fail. It
  does not roll back. It just stops, forever.

`kubectl describe pod <the-broken-one> -n k8s-lab` would show
`Failed to pull image "k8s-lab/user-service:does-not-exist"` in its Events.

Now undo it.

## ⏪ Rollback — What It Actually Does

`kubectl rollout undo` is not magic and it is not a time machine. Here is the real
mechanism, because the mental model matters:

Every time you change a Deployment's **pod template**, Kubernetes creates a new
**ReplicaSet** and scales the old one down to zero. It does not delete the old
ReplicaSet. `kubectl rollout undo` simply scales the previous ReplicaSet back up and the
current one back down.

Consequences worth internalising:

- **Rollback is fast** — the old ReplicaSet's pod template is already stored in the
  cluster. Nothing is rebuilt, nothing is re-fetched from Git.
- **Rollback restores the pod template and nothing else.** ConfigMaps, Secrets, database
  schema migrations, and anything else your release touched are *not* reverted. If your
  bad release ran a migration, undoing the Deployment leaves you running old code against
  a new schema. This is the number one way "just roll it back" goes wrong in production.
- **History is bounded.** `spec.revisionHistoryLimit` (default `10`) caps how many old
  ReplicaSets are kept. Older revisions are garbage-collected and can no longer be
  rolled back to.
- Scaling is *not* a revision. `kubectl scale` does not touch the pod template, so it
  creates no new ReplicaSet and does not show up in `rollout history`.

### Exercise

Undo the failed rollout and confirm the Deployment becomes healthy again.

In [ ]:
!kubectl rollout undo deployment/user-service -n k8s-lab
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=180s
!kubectl get pods -l app=user-service -n k8s-lab

print()
# One ReplicaSet per pod-template revision. The ones at 0 desired are the old
# revisions kept around so `rollout undo` has somewhere to go back to.
!kubectl get replicaset -l app=user-service -n k8s-lab
!kubectl rollout history deployment/user-service -n k8s-lab

healthy = settled_pods("app=user-service", 3)
assert all(c["image"] == "k8s-lab/user-service:v2"
           for pod in healthy for c in pod["spec"]["containers"]), \
    "undo did not restore the v2 image"

# The mechanism, made visible: several ReplicaSets exist, only one is scaled up.
rs = kget("replicaset", "-l", "app=user-service")["items"]
scaled_up = [r for r in rs if r["spec"]["replicas"]]
assert len(rs) >= 2, "rollback needs an older ReplicaSet to scale back up"
assert len(scaled_up) == 1, f"exactly one ReplicaSet should be active, found {len(scaled_up)}"
print(f"\n✅ rolled back: {len(rs)} ReplicaSets exist, 1 scaled to 3, image is v2")

## 📏 Resource Requests and Limits

This is the section people skim and then get paged about. Requests and limits do
**completely different jobs**, and CPU and memory behave **completely differently** when
you exceed a limit.

### Request = a scheduling promise

`requests` is the *only* number the **scheduler** looks at. When it picks a node, it adds
up the requests of everything already on that node and asks "does this new pod's request
still fit in the node's allocatable capacity?" It does not look at limits, and it does not
look at what the pods are *actually* using.

Two consequences:

- **A pod with no requests is free from the scheduler's point of view.** Ten such pods
  will happily be packed onto one node until the kubelet starts evicting things under
  memory pressure and everything on that node gets slow and unreliable together. This is
  the classic "my cluster has plenty of free CPU but is falling over" incident.
- **Over-requesting wastes money.** Nodes fill up on paper while sitting at 10% real
  usage.

### Limit = a runtime ceiling — enforced very differently per resource

| Resource | Over the limit → | What you observe |
|---|---|---|
| **CPU** | **Throttled** | The container is *not* killed. The Linux CFS scheduler simply gives it less CPU time for the rest of each 100 ms period. Your p99 latency climbs; nothing appears in the pod's status. Look for `container_cpu_cfs_throttled_seconds_total`. |
| **Memory** | **OOMKilled** | There is no such thing as "throttled memory". The kernel OOM-killer terminates the process immediately. `kubectl get pods` shows a restart; `kubectl describe pod` shows `Last State: Terminated, Reason: OOMKilled, Exit Code: 137`. |

That asymmetry drives a widely-used production rule of thumb:

> **Always set a memory limit equal to the memory request. Be cautious with CPU limits.**

Memory is incompressible — if you let a container burst above its request there is nothing
to reclaim when the node runs out, so it is better to fail one pod predictably than to
destabilise a whole node. CPU is compressible — an unlimited container that briefly uses
spare CPU harms no one, whereas a tight CPU limit throttles a healthy service that had
idle CPU sitting right next to it.

### QoS class — derived, never set by you

Kubernetes reads your requests and limits and stamps a **QoS class** onto the pod. You
cannot set it directly; you can only cause it.

| QoS class | Condition | What it buys you |
|---|---|---|
| **Guaranteed** | *Every* container in the pod sets **both** requests and limits for **both** CPU and memory, and for each resource `request == limit` | Evicted **last** under node pressure. Lowest OOM score. |
| **Burstable** | At least one container sets a request or a limit, but the pod does not qualify as Guaranteed | Evicted in the middle, worst offenders (usage furthest above request) first. |
| **BestEffort** | **No** container sets any request or limit at all | Evicted **first**, always. Never use this for anything you care about. |

Note the strictness: setting `requests.cpu` only, or setting `limits.memory` to a
slightly different value than `requests.memory`, drops the pod from Guaranteed to
Burstable. Notebook 10 has you create one pod of each class and read the class back out
of the API.

### Exercise

Add CPU and memory requests and limits to the Deployment, then read back the QoS class
Kubernetes derived.

In [ ]:
# requests == limits for BOTH cpu and memory  ->  QoS class Guaranteed
!kubectl set resources deployment user-service -n k8s-lab --requests=cpu=100m,memory=128Mi --limits=cpu=100m,memory=128Mi
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=80s

print()
# We never set qosClass -- the API server derived it from the numbers above.
!kubectl get pods -l app=user-service -n k8s-lab -o custom-columns=NAME:.metadata.name,QOS:.status.qosClass

classes = {p["status"]["qosClass"] for p in settled_pods("app=user-service", 3)}
assert classes == {"Guaranteed"}, \
    f"request == limit on both resources should give Guaranteed, got {classes}"

print()
# Now make the limits differ from the requests and watch the class drop to Burstable.
!kubectl set resources deployment user-service -n k8s-lab --requests=cpu=100m,memory=128Mi --limits=cpu=250m,memory=256Mi
!kubectl rollout status deployment/user-service -n k8s-lab --timeout=80s
!kubectl get pods -l app=user-service -n k8s-lab -o custom-columns=NAME:.metadata.name,QOS:.status.qosClass

classes = {p["status"]["qosClass"] for p in settled_pods("app=user-service", 3)}
assert classes == {"Burstable"}, \
    f"limits != requests should demote the pod to Burstable, got {classes}"
print("\n✅ Guaranteed -> Burstable, caused only by changing the numbers")

## ❤️ Health Probes: Liveness vs Readiness vs Startup

Three probes, three completely different jobs. Confusing them is one of the few
Kubernetes mistakes that makes your service *less* reliable than having no probes at all,
so it is worth being precise.

| Probe | Question it answers | Action on failure | Effect on traffic |
|---|---|---|---|
| **readiness** | "Can this pod serve requests **right now**?" | Pod's IP is removed from the Service's Endpoints | Traffic stops. Container keeps running. |
| **liveness** | "Is this container **wedged** and only a restart will fix it?" | kubelet **kills and restarts the container** | Indirect — the pod is gone while it restarts |
| **startup** | "Has this slow-starting container finished booting?" | Container is killed | Suspends liveness *and* readiness until it first succeeds |

### The rules that matter

**1. Readiness is the one that controls traffic. Liveness never does.**
A failing liveness probe does not take a pod out of load balancing — it restarts the
container. If you only configure liveness, then during the seconds between "container
process started" and "app can actually serve", the pod is already in the Service's
Endpoints and is receiving (and dropping) real requests. This is exactly the gap that
`maxUnavailable: 0` was supposed to protect you from, and without a readiness probe it
does not.

**2. Never point a liveness probe at a dependency.**
If `/health` checks the database and the database blips, *every* replica fails liveness
at the same instant, *every* replica is restarted at the same instant, and you have
converted a recoverable dependency outage into a full self-inflicted outage with cold
caches. A liveness probe must test only "is this process still able to make progress?" —
ideally a trivial handler that does nothing but return 200. Put dependency checks in the
**readiness** probe, where the consequence is "stop sending me traffic until it recovers"
rather than "kill me".

**3. Liveness should be lazier than readiness.**
Give liveness a longer `periodSeconds` and a higher `failureThreshold`. A restart is a
big hammer; you want to be sure.

**4. Slow starts are a `startupProbe`, not a big `initialDelaySeconds`.**
`initialDelaySeconds: 120` also delays detection of a genuine hang by two minutes,
forever. A `startupProbe` with `failureThreshold: 30, periodSeconds: 5` gives the app up
to 150 seconds to boot and then hands over to a *fast* liveness probe. Best of both.

### Failure first

The cell below deploys `user-service` with **no readiness probe** but with a
`sleep`-delayed container, and shows the pod being reported as ready — and therefore
added to Service endpoints — while it is still incapable of answering. Then we add the
readiness probe and watch the behaviour change.

In [ ]:
%%writefile ./probe-demo.yaml
apiVersion: v1
kind: Service
metadata:
  name: probe-demo
  namespace: k8s-lab
spec:
  selector:
    app: probe-demo
  ports:
    - name: http
      port: 80
      targetPort: 80
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: probe-demo
  namespace: k8s-lab
spec:
  replicas: 1
  selector:
    matchLabels:
      app: probe-demo
  template:
    metadata:
      labels:
        app: probe-demo
    spec:
      containers:
        - name: web
          image: nginx:1.27
          ports:
            - containerPort: 80
          # Simulate a slow-starting app: the container process (sh) starts
          # instantly, but nothing is listening on port 80 for 45 seconds.
          command: ["sh", "-c", "sleep 45 && exec nginx -g 'daemon off;'"]

In [ ]:
# Start from a clean slate so a re-run of this notebook does not leave the
# previous (probe-equipped) pod around while we look at the probe-less one.
!kubectl delete -f ./probe-demo.yaml --ignore-not-found
!kubectl apply -f ./probe-demo.yaml

# Sample the moment the container process starts, NOT after some fixed sleep --
# an image pull on a cold node would otherwise push the interesting window past
# a hard-coded `time.sleep(8)` and the demo would show nothing.
def running_probe_demo():
    for pod in pods("app=probe-demo"):
        if pod["status"].get("phase") == "Running":
            return pod
    return None


started = wait_until(running_probe_demo, timeout=180, interval=2,
                     what="the probe-demo container to start")

print("\n--- pod status the instant the container process started (no readiness probe) ---")
!kubectl get pods -l app=probe-demo -n k8s-lab

print("\n--- Service endpoints: is this pod already receiving traffic? ---")
!kubectl get endpoints probe-demo -n k8s-lab

# The lesson, asserted: nothing is listening on port 80 for another ~45 seconds,
# and Kubernetes is routing traffic to it anyway.
conds = {c["type"]: c["status"] for c in started["status"].get("conditions", [])}
assert conds.get("Ready") == "True", (
    "the point of this demo is a pod reported Ready before it can serve; it is not "
    f"Ready here, so something else is wrong: {conds}"
)
ips = wait_until(lambda: endpoint_ips("probe-demo"), timeout=30, interval=2,
                 what="the not-yet-serving pod to appear in Service endpoints")
assert started["status"]["podIP"] in ips, \
    f"expected the pod IP in the Service endpoints, got {ips}"
print(f"\n✅ reproduced: pod is 'Ready' and in endpoints {ips} while nothing listens on :80")

The pod shows `1/1  Running` and its IP is **already listed in the Service's
endpoints** — even though nothing will answer on port 80 for another ~45 seconds. Every
request routed there right now fails with a connection refused.

Without a readiness probe, "Ready" means nothing more than "the container's PID 1 is
alive". Here that PID is `sh`, sleeping. Now add a probe.

In [ ]:
%%writefile ./probe-demo.yaml
apiVersion: v1
kind: Service
metadata:
  name: probe-demo
  namespace: k8s-lab
spec:
  selector:
    app: probe-demo
  ports:
    - name: http
      port: 80
      targetPort: 80
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: probe-demo
  namespace: k8s-lab
spec:
  replicas: 1
  selector:
    matchLabels:
      app: probe-demo
  template:
    metadata:
      labels:
        app: probe-demo
    spec:
      containers:
        - name: web
          image: nginx:1.27
          ports:
            - containerPort: 80
          command: ["sh", "-c", "sleep 45 && exec nginx -g 'daemon off;'"]

          # startupProbe: give a slow boot up to 30 x 5s = 150s. Liveness and
          # readiness are both suspended until this succeeds once.
          startupProbe:
            httpGet:
              path: /
              port: 80
            periodSeconds: 5
            failureThreshold: 30

          # readinessProbe: gates Service endpoints. Cheap, frequent, and it is
          # allowed to fail without anything being killed.
          readinessProbe:
            httpGet:
              path: /
              port: 80
            periodSeconds: 5
            failureThreshold: 2

          # livenessProbe: restarts a wedged container. Deliberately lazier than
          # readiness, and it must NOT check downstream dependencies.
          livenessProbe:
            httpGet:
              path: /
              port: 80
            periodSeconds: 15
            failureThreshold: 3

In [ ]:
!kubectl apply -f ./probe-demo.yaml

# NOTE the ordering. `kubectl rollout status` only returns once the new pod is
# Ready, so running it first would hide exactly the window we came to look at.
# Find the new (probe-equipped) pod, then measure it while it is still booting.
def new_pod():
    for pod in pods("app=probe-demo"):
        spec = pod["spec"]["containers"][0]
        if "startupProbe" in spec and pod["status"].get("phase") == "Running":
            return pod
    return None


booting = wait_until(new_pod, timeout=120, interval=2,
                     what="the probe-equipped pod's container to start")

print("\n--- while the new pod is still booting: NOT ready, and NOT in endpoints ---")
!kubectl get pods -l app=probe-demo -n k8s-lab
!kubectl get endpoints probe-demo -n k8s-lab

conds = {c["type"]: c["status"] for c in booting["status"].get("conditions", [])}
assert conds.get("Ready") != "True", \
    "the startup probe should be holding this pod out of Ready while it boots"
assert booting["status"]["podIP"] not in endpoint_ips("probe-demo"), \
    "a pod that cannot serve yet must not be in the Service's endpoints"
print("✅ the probe is holding it out of load balancing")

In [ ]:
# The app sleeps for 45 seconds before it binds port 80, so this is the wait.
# Note what we do NOT run here: `kubectl wait --for=condition=Ready pod -l
# app=probe-demo`. A label selector matches the OLD pod too, and once the rollout
# scales it down, kubectl is left waiting on a pod that is being deleted and never
# becomes Ready -- a hang that looks like a broken probe and is not.
print("--- waiting for the app to finish booting ---")
!kubectl rollout status deployment/probe-demo -n k8s-lab --timeout=150s

serving = settled_pods("app=probe-demo", 1, timeout=150)
!kubectl get pods -l app=probe-demo -n k8s-lab
!kubectl get endpoints probe-demo -n k8s-lab

assert serving[0]["status"]["podIP"] in endpoint_ips("probe-demo"), \
    "once Ready, the pod should be back in the Service's endpoints"
print("\n✅ and once it can actually serve, it is added to endpoints")

Now the pod reports `0/1` and the Service's endpoint list is `<none>` until the app can
genuinely serve. `kubectl rollout status` waits for the same signal, so rolling updates
became safe at the same moment — one probe fixed both.

An empty `ENDPOINTS` column is also the single best diagnostic in Kubernetes networking:
if a Service returns connection refused, run `kubectl get endpoints <svc>` first. Empty
means either no pod matches the Service's selector, or every matching pod is failing
readiness. Notebook 3 goes deeper on the selector half of that.

## 🧹 Clean Up — and Hand Off to Notebook 3

Two different things happen here, and it is worth knowing which is which.

**Removed**: the throwaway objects this notebook invented — `hello-pod`, `probe-demo`,
and the YAML files we generated beside the notebook.

**Kept and completed**: the `k8s-lab` namespace and the three sample services. Notebooks
3 to 10 all build on them, so instead of deleting `user-service` we apply the shared
manifests in `../manifests/`, which bring `api-gateway` and `order-service` alongside it
and reconcile `user-service` to its production-shaped definition — 2 replicas, requests
and limits, and readiness/liveness probes on all three.

Because everything uses `apply` and `--ignore-not-found`, this cell is safe to run twice,
and the whole notebook is safe to re-run from the top.

### Exercise

Clean up the scratch objects and deploy the shared lab services.

In [ ]:
# 1. Remove this notebook's throwaway objects.
!kubectl delete pod hello-pod -n k8s-lab --ignore-not-found
!kubectl delete -f ./probe-demo.yaml --ignore-not-found
!rm -f ./pod.yaml ./user-service-deployment.yaml ./probe-demo.yaml

print()
# 2. Deploy the shared lab services that notebooks 3-10 depend on.
#    This also reconciles user-service to the manifest version (2 replicas,
#    requests/limits, readiness probe), replacing what we hand-built above.
!kubectl apply -f ../manifests/configmap.yaml
!kubectl apply -f ../manifests/deployment.yaml
# Three `kubectl rollout status --timeout=180s` calls in a row can add up to nine
# minutes in the worst case, which is longer than most notebook runners allow a
# single cell. Wait on the state we actually need instead, with one budget.
DEPLOYMENTS = ("api-gateway", "user-service", "order-service")


def all_ready():
    counts = {n: kget("deployment", n)["status"].get("readyReplicas", 0)
              for n in DEPLOYMENTS}
    print("  ready replicas:", counts)
    return counts if all(v == 2 for v in counts.values()) else None


wait_until(all_ready, timeout=170, interval=10,
           what="all three shared deployments to reach 2/2")

print()
!kubectl get deployments -n k8s-lab
print("\n✅ handed off to notebook 3: api-gateway, user-service, order-service at 2/2")

## 🎓 What You Learned

In this notebook, you:
- Created a pod from YAML in the `k8s-lab` namespace
- Built the three sample images directly into minikube
- Created a Deployment and saw the Deployment → ReplicaSet → Pod chain
- Scaled replicas up to 3
- Ran a rolling update with `maxSurge: 1, maxUnavailable: 0`
- **Broke a rollout on purpose** and watched Kubernetes protect the running version
- Learned what `rollout undo` really does (scales the previous ReplicaSet back up) and
  what it does *not* do (revert ConfigMaps, Secrets, or database migrations)
- Set requests and limits, and derived the pod's **QoS class** from them
- Learned that exceeding a **CPU** limit throttles, while exceeding a **memory** limit
  **OOMKills** — and why that asymmetry drives the "memory limit == memory request"
  rule of thumb
- Configured **startup**, **readiness** and **liveness** probes, and saw a pod being sent
  traffic before it could serve it when the readiness probe was missing

### Left running for the next notebooks

`k8s-lab` now contains `api-gateway`, `user-service` and `order-service`, each with
2 replicas. Notebook 3 puts Services and an Ingress in front of them.